<a href="https://colab.research.google.com/github/CSCEJJQ/344Lab/blob/main/notebooks/04_what_it_learned.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [54]:
USER, REPO = "CSCEJJQ", "344Lab"
!rm -rf {REPO}
!git clone -q https://github.com/{USER}/{REPO}.git
import sys
sys.path.insert(0, f"{REPO}/src")
from evaluate import to_matrix
from models import baseline_matrix, rank_k

This clones the gihub repo and imports functions from the python files

In [55]:
import os
import urllib.request
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if not os.path.exists("ml-100k"):
    urllib.request.urlretrieve(
        "https://files.grouplens.org/datasets/movielens/ml-100k.zip",
        "ml-100k.zip")
    zipfile.ZipFile("ml-100k.zip").extractall()

titles = pd.read_csv("ml-100k/u.item", sep="|", encoding="latin-1",
header=None, usecols=[1])[1].to_numpy()

ratings = pd.read_csv(
    "ml-100k/u.data", sep="\t",
    names=["user", "item", "rating", "time"])

print(len(titles), titles[49])

1682 Star Wars (1977)


This cell donwloads and reads the movie file

In [56]:
R = to_matrix(ratings)
known = ~np.isnan(R)
B = baseline_matrix(R)
X = np.where(known, R - B, 0.0)
U, s, Vt = np.linalg.svd(X, full_matrices=False)
n_ratings = known.sum(axis=0)
popular = n_ratings >= 50
print("popular movies:", popular.sum())

popular movies: 603


This cell performs the svd on the ratings matrix

In [57]:
def show_factor(f, n=6):
  """Popular movies at the two ends of hidden factor f (f = 1, 2, ...)."""
  v = Vt[f - 1]
  idx = np.where(popular)[0]
  order = idx[np.argsort(v[idx])]
  print(f"Factor {f}, one end:")
  for m in order[:n]:
    print(f" {v[m]:+.3f} {titles[m]}")
  print(f"Factor {f}, other end:")
  for m in order[::-1][:n]:
    print(f" {v[m]:+.3f} {titles[m]}")

show_factor(1)

Factor 1, one end:
 -0.163 Pulp Fiction (1994)
 -0.136 Clockwork Orange, A (1971)
 -0.123 Trainspotting (1996)
 -0.123 Fargo (1996)
 -0.117 Leaving Las Vegas (1995)
 -0.106 Brazil (1985)
Factor 1, other end:
 +0.206 Independence Day (ID4) (1996)
 +0.127 Return of the Jedi (1983)
 +0.123 Back to the Future (1985)
 +0.120 Raiders of the Lost Ark (1981)
 +0.114 Indiana Jones and the Last Crusade (1989)
 +0.111 Jurassic Park (1993)


This cell displays movies by their SVD factor

In [58]:
def find(word):
  for m, t in enumerate(titles):
    if word.lower() in t.lower():
      print(m+1, t)

find("evil")

201 Evil Dead II (1987)
245 Devil's Own, The (1997)
307 Devil's Advocate, The (1997)
312 Midnight in the Garden of Good and Evil (1997)
653 Touch of Evil (1958)
770 Devil in a Blue Dress (1995)
1456 Beat the Devil (1954)


This cell defines a function that allows searching by a title

In [59]:
print(titles[256])
print(titles[958])
print(titles[894])
print(titles[819])
print(titles[754])
print(titles[671])
print(titles[450])
print(titles[418])
print(titles[422])
print(titles[200])

Men in Black (1997)
Dazed and Confused (1993)
Scream 2 (1997)
Space Jam (1996)
Jumanji (1995)
Candyman (1992)
Grease (1978)
Mary Poppins (1964)
E.T. the Extra-Terrestrial (1982)
Evil Dead II (1987)


In [60]:
my_ratings = {257: 5, 959: 2, 895: 5, 820: 2, 755: 3, 672: 5, 451: 2, 419: 1,
              423: 2, 201: 5}
opposite_ratings = {257: 1, 959: 4, 895: 1, 820: 4, 755: 3, 672: 1, 451: 4, 419: 5,
              423: 4, 201: 1}

In [61]:
me = np.full((1, R.shape[1]), np.nan)
for movie, stars in my_ratings.items():
  me[0, movie - 1] = stars
R_me = np.vstack([R, me])
print(R_me.shape)

known_me = ~np.isnan(R_me)
B_me = baseline_matrix(R_me)
X_me = np.where(known_me, R_me - B_me, 0.0)
U, s, Vt = np.linalg.svd(X_me, full_matrices=False)
P = rank_k(U, s, Vt, 10) + B_me

(944, 1682)


In [62]:
mine = P[-1]
base = B_me[-1]
can_pick = popular & np.isnan(R_me[-1])

def top(scores, n=10):
  """The n highest-scoring movies among those we may recommend."""
  return np.argsort(-np.where(can_pick, scores, -np.inf))[:n]

print("Model 2 recommends:")
for m in top(mine):
  print(f" {mine[m]:.2f} {titles[m]}")
print("Baseline recommends:")
for m in top(base):
  print(f" {base[m]:.2f} {titles[m]}")
print("in both lists:", len(set(top(mine)) & set(top(base))))

Model 2 recommends:
 4.35 Wrong Trousers, The (1993)
 4.34 Close Shave, A (1995)
 4.30 Wallace & Gromit: The Best of Aardman Animation (1996)
 4.25 Casablanca (1942)
 4.24 Schindler's List (1993)
 4.20 Shawshank Redemption, The (1994)
 4.20 Rear Window (1954)
 4.19 Usual Suspects, The (1995)
 4.18 Third Man, The (1949)
 4.16 12 Angry Men (1957)
Baseline recommends:
 4.32 Close Shave, A (1995)
 4.30 Schindler's List (1993)
 4.30 Wrong Trousers, The (1993)
 4.29 Casablanca (1942)
 4.28 Wallace & Gromit: The Best of Aardman Animation (1996)
 4.28 Shawshank Redemption, The (1994)
 4.22 Rear Window (1954)
 4.22 Usual Suspects, The (1995)
 4.19 Star Wars (1977)
 4.18 12 Angry Men (1957)
in both lists: 9


In [63]:
personal = mine - base

print("Moved up most for me:")
for m in top(personal, 6):
  print(f" {personal[m]:+.2f} {titles[m]}")
print("Moved down most for me:")
for m in top(-personal, 6):
  print(f" {personal[m]:+.2f} {titles[m]}")

Moved up most for me:
 +0.17 Scream (1996)
 +0.13 Pulp Fiction (1994)
 +0.12 Twelve Monkeys (1995)
 +0.12 Rock, The (1996)
 +0.10 Starship Troopers (1997)
 +0.10 Army of Darkness (1993)
Moved down most for me:
 -0.13 Sound of Music, The (1965)
 -0.11 Wizard of Oz, The (1939)
 -0.11 Field of Dreams (1989)
 -0.11 Dumbo (1941)
 -0.10 Snow White and the Seven Dwarfs (1937)
 -0.10 To Kill a Mockingbird (1962)


Without being told any of the genres, the model looks at the movies I like and moves up movies like scream and army of darkness and moves down movies like dumbo and snow white. It can tell from the movies I like that I prefer darker movies to family animations.

Exercises:
1. I think roughly factor 1 is thrillers, factor 2 is action movies, factor 3 is adventure movies. I think factor 2 is harder to name since some of the movies on the bottom end like jaws and star wars I would classify more as action movies.
2. 9/10 of the movies recommended by model 2 are in the baseline recomendation. I think the moves up and moved down lists are accurate, but I don't think the model should recommend a movie like wallace and gromit based on my movie ratings.
3. The baseline recommendations did not change much, but movies like twelve monkeys, the rock, and army of darkness went from being moved up to being moved down; movie likes snow white and the sound of music went from being moved down to moved up.
4. The moved up list changes movies and when run with k=50 it greatly increases the amount that each movie is moved up. I think with k=50 it is overfitting because it reads too much into the data and finds patterns that are not there, since none of the movies that were moved up seem particularly similar to the ones that were moved up with k=10.

In [64]:
show_factor(1)
print("-"*40)
show_factor(2)
print("-"*40)
show_factor(3)

Factor 1, one end:
 -0.163 Pulp Fiction (1994)
 -0.136 Clockwork Orange, A (1971)
 -0.123 Trainspotting (1996)
 -0.123 Fargo (1996)
 -0.117 Leaving Las Vegas (1995)
 -0.106 Brazil (1985)
Factor 1, other end:
 +0.205 Independence Day (ID4) (1996)
 +0.127 Return of the Jedi (1983)
 +0.123 Back to the Future (1985)
 +0.120 Raiders of the Lost Ark (1981)
 +0.114 Indiana Jones and the Last Crusade (1989)
 +0.111 Jurassic Park (1993)
----------------------------------------
Factor 2, one end:
 -0.115 Independence Day (ID4) (1996)
 -0.092 Twister (1996)
 -0.079 Saint, The (1997)
 -0.073 Air Force One (1997)
 -0.069 Rock, The (1996)
 -0.062 Murder at 1600 (1997)
Factor 2, other end:
 +0.180 Star Wars (1977)
 +0.144 Jaws (1975)
 +0.141 Empire Strikes Back, The (1980)
 +0.137 Scream (1996)
 +0.136 Godfather, The (1972)
 +0.125 Pulp Fiction (1994)
----------------------------------------
Factor 3, one end:
 -0.247 Star Wars (1977)
 -0.202 Return of the Jedi (1983)
 -0.170 Empire Strikes Back, The